In [8]:
# ==============================================================================
# SKRIP REVISI MAYOR SINKRON (MENJAWAB POIN 1, 3, 4, DAN 5 SEKALIGUS)
# ==============================================================================
import time
import warnings
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# 1. LOAD DATASET
FINAL_DATA_FILE = "CIC-IDS2017_20features_clean.csv"
df = pd.read_csv(FINAL_DATA_FILE)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

selected_features = [
    "Avg Packet Size",
    "Bwd Packet Length Mean",
    "Avg Bwd Segment Size",
    "Subflow Bwd Bytes",
    "Bwd Packets Length Total",
    "Bwd Packet Length Max",
    "Packet Length Std",
    "Packet Length Variance",
    "Fwd IAT Max",
    "Packet Length Mean",
    "Fwd IAT Total",
    "Fwd Packets Length Total",
    "Subflow Fwd Bytes",
    "Packet Length Max",
    "Bwd Packet Length Std",
    "Init Bwd Win Bytes",
    "Fwd Packet Length Max",
    "Init Fwd Win Bytes",
    "Fwd IAT Mean",
    "Fwd Packet Length Std",
]

X = df[selected_features].astype(np.float32).copy()
y = df["Label"].astype(np.int8).copy()

# Buat Group ID untuk Duplicate-Aware Split (786,885 unique groups)
groups = pd.factorize(pd.util.hash_pandas_object(X, index=False))[0]
groups = pd.Series(groups, index=X.index)


# Fungsi Pembantu: Model & Sample Weight
def create_rf(seed=2024, class_weight=None):
    return RandomForestClassifier(
        n_estimators=100, random_state=seed, n_jobs=1, class_weight=class_weight
    )


def create_xgb(seed=2024):
    return XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=1,
    )


def get_balanced_weights(y_tr):
    counts = y_tr.value_counts()
    n_samples = len(y_tr)
    weights = {cls: n_samples / (2.0 * cnt) for cls, cnt in counts.items()}
    return y_tr.map(weights).values


def apply_handling(X_tr, y_tr, handling, seed=2024):
    t0 = time.time()
    if handling in ["None", "Class Weight"]:
        X_res, y_res = X_tr, y_tr
    elif handling == "SMOTE":
        X_res, y_res = SMOTE(random_state=seed).fit_resample(X_tr, y_tr)
    elif handling == "ADASYN":
        X_res, y_res = ADASYN(random_state=seed).fit_resample(X_tr, y_tr)
    elif handling == "RUS":
        X_res, y_res = RandomUnderSampler(random_state=seed).fit_resample(
            X_tr, y_tr
        )
    elif handling == "Tomek Links":
        X_res, y_res = TomekLinks().fit_resample(X_tr, y_tr)
    resample_time = time.time() - t0
    return X_res, y_res, resample_time


def eval_full(
    model, X_te, y_te, model_name, handling, split_type, resample_time, fit_time
):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    fpr = fp / (fp + tn)
    return {
        "Split_Type": split_type,
        "Model": model_name,
        "Handling": handling,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "FPR": fpr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "ROC_AUC": roc_auc_score(y_te, y_prob),
        "AP": average_precision_score(y_te, y_prob),
        "Resample_Time_s": resample_time,
        "Fit_Time_s": fit_time,
        "Total_Time_s": resample_time + fit_time,
    }


print("Setup selesai. Siap menjalankan eksperimen revisi.")

Setup selesai. Siap menjalankan eksperimen revisi.


In [9]:
handlings = ["None", "Class Weight", "SMOTE", "ADASYN", "RUS", "Tomek Links"]

# Contoh: ambil satu fold StratifiedGroupKFold sebagai Fold 2
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2024)
splits = list(sgkf.split(X, y, groups))
tr_idx, te_idx = splits[1]  # Fold 2 (0-indexed)
X_train_da, X_test_da = X.iloc[tr_idx], X.iloc[te_idx]
y_train_da, y_test_da = y.iloc[tr_idx], y.iloc[te_idx]

In [10]:
# Deteksi group berlabel konflik
tmp = pd.DataFrame({"g": groups, "y": y})
conflict_groups = tmp.groupby("g")["y"].nunique()
conflict_ids = conflict_groups[conflict_groups > 1].index
mask_clean = ~groups.isin(conflict_ids)
X_clean, y_clean, groups_clean = X[mask_clean], y[mask_clean], groups[mask_clean]
print("Conflicting groups removed:", len(conflict_ids))

Conflicting groups removed: 91


In [11]:
# ==============================================================================
# EKSPERIMEN REVISI 1: SEMUA KONFIGURASI PADA DUPLICATE-AWARE SPLIT
# ==============================================================================

# Pastikan conflict groups sudah dihapus SEBELUM cell ini
# (X_clean, y_clean, groups_clean sudah tersedia)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2024)
splits = list(sgkf.split(X_clean, y_clean, groups_clean))

# Pilih satu fold (misal index 2)
train_idx_da, test_idx_da = splits[2]
X_train_da = X_clean.iloc[train_idx_da].copy()
X_test_da  = X_clean.iloc[test_idx_da].copy()
y_train_da = y_clean.iloc[train_idx_da].copy()
y_test_da  = y_clean.iloc[test_idx_da].copy()

# Cek tidak ada group yang bocor
shared = set(groups_clean.iloc[train_idx_da]) & set(groups_clean.iloc[test_idx_da])
print("Shared groups (harus 0):", len(shared))
print("Train size:", len(X_train_da), "| Test size:", len(X_test_da))

handlings = ["None", "Class Weight", "SMOTE", "ADASYN", "RUS", "Tomek Links"]
rev1_results = []

for h in handlings:
    print(f"Memproses Duplicate-Aware Split | Handling: {h} ...")
    X_res, y_res, r_time = apply_handling(X_train_da, y_train_da, h, seed=2024)

    # 1. Random Forest
    cw = "balanced" if h == "Class Weight" else None
    rf = create_rf(seed=2024, class_weight=cw)
    t_fit0 = time.time()
    rf.fit(X_res, y_res)
    rf_fit_time = time.time() - t_fit0
    rev1_results.append(
        eval_full(
            rf, X_test_da, y_test_da,
            "Random Forest", h, "Duplicate-Aware (Fold 2)",
            r_time, rf_fit_time,
        )
    )

    # 2. XGBoost
    xgb = create_xgb(seed=2024)
    sw = get_balanced_weights(pd.Series(y_res)) if h == "Class Weight" else None
    t_fit0 = time.time()
    xgb.fit(X_res, y_res, sample_weight=sw)
    xgb_fit_time = time.time() - t_fit0
    rev1_results.append(
        eval_full(
            xgb, X_test_da, y_test_da,
            "XGBoost", h, "Duplicate-Aware (Fold 2)",
            r_time, xgb_fit_time,
        )
    )

rev1_df = pd.DataFrame(rev1_results)
rev1_df.to_csv("Revisi_Duplicate_Aware_All_Configs.csv", index=False)
print("\n=== HASIL DUPLICATE-AWARE SPLIT (SEMUA KONFIGURASI) ===")
display(rev1_df.round(6))

Shared groups (harus 0): 0
Train size: 993663 | Test size: 255319
Memproses Duplicate-Aware Split | Handling: None ...
Memproses Duplicate-Aware Split | Handling: Class Weight ...
Memproses Duplicate-Aware Split | Handling: SMOTE ...


  File "C:\Users\brene\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Memproses Duplicate-Aware Split | Handling: ADASYN ...
Memproses Duplicate-Aware Split | Handling: RUS ...
Memproses Duplicate-Aware Split | Handling: Tomek Links ...

=== HASIL DUPLICATE-AWARE SPLIT (SEMUA KONFIGURASI) ===


,Split_Type,Model,Handling,Accuracy,Precision,Recall,F1,FPR,TN,FP,FN,TP,ROC_AUC,AP,Resample_Time_s,Fit_Time_s,Total_Time_s
0,Duplicate-Aware (Fold 2),Random Forest,None,0.999792,0.999609,0.999562,0.999586,0.000131,191318,25,28,63948,0.999942,0.999901,0.000000,133.505301,133.505301
1,Duplicate-Aware (Fold 2),XGBoost,None,0.999800,0.999562,0.999640,0.999601,0.000146,191315,28,23,63953,0.999999,0.999996,0.000000,20.428726,20.428726
2,Duplicate-Aware (Fold 2),Random Forest,Class Weight,0.999816,0.999656,0.999609,0.999633,0.000115,191321,22,25,63951,0.999957,0.999924,0.000000,131.470567,131.470567
3,Duplicate-Aware (Fold 2),XGBoost,Class Weight,0.999788,0.999437,0.999719,0.999578,0.000188,191307,36,18,63958,0.999999,0.999996,0.000000,20.127833,20.127833
4,Duplicate-Aware (Fold 2),Random Forest,SMOTE,0.999722,0.999250,0.999640,0.999445,0.000251,191295,48,23,63953,0.999939,0.999885,42.380434,249.132927,291.513361
5,Duplicate-Aware (Fold 2),XGBoost,SMOTE,0.999730,0.999188,0.999734,0.999461,0.000272,191291,52,17,63959,0.999997,0.999993,42.380434,30.406264,72.786697
6,Duplicate-Aware (Fold 2),Random Forest,ADASYN,0.999561,0.998626,0.999625,0.999125,0.000460,191255,88,24,63952,0.999948,0.999873,220.714068,256.067626,476.781695
7,Duplicate-Aware (Fold 2),XGBoost,ADASYN,0.999252,0.997178,0.999844,0.998509,0.000946,191162,181,10,63966,0.999997,0.999990,220.714068,30.289975,251.004043
8,Duplicate-Aware (Fold 2),Random Forest,RUS,0.999620,0.998782,0.999703,0.999242,0.000408,191265,78,19,63957,0.999957,0.999923,0.128028,49.673854,49.801882
9,Duplicate-Aware (Fold 2),XGBoost,RUS,0.999722,0.999110,0.999781,0.999445,0.000298,191286,57,14,63962,0.999998,0.999995,0.128028,10.408674,10.536703


In [3]:
# ==============================================================================
# EKSPERIMEN REVISI 2: UJI SENSITIVITAS SCALING (POIN 3) & MULTI-FOLD (POIN 4)
# ==============================================================================

# A. UJI SENSITIVITAS STANDARD SCALER PADA SMOTE, ADASYN, TOMEK LINKS (XGBoost & RF)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_da),
    columns=selected_features,
    index=X_train_da.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_da),
    columns=selected_features,
    index=X_test_da.index,
)

scaling_results = []
for h in ["SMOTE", "ADASYN", "Tomek Links"]:
    print(f"Memproses Uji Scaling | Handling: {h} ...")
    X_res_sc, y_res_sc, r_time_sc = apply_handling(
        X_train_scaled, y_train_da, h, seed=2024
    )

    # XGBoost dengan Scaled Resampling
    xgb_sc = create_xgb(seed=2024)
    t0 = time.time()
    xgb_sc.fit(X_res_sc, y_res_sc)
    fit_t = time.time() - t0
    scaling_results.append(
        eval_full(
            xgb_sc,
            X_test_scaled,
            y_test_da,
            "XGBoost",
            h,
            "Scaled + Duplicate-Aware",
            r_time_sc,
            fit_t,
        )
    )

scaling_df = pd.DataFrame(scaling_results)
scaling_df.to_csv("Revisi_Scaling_Sensitivity.csv", index=False)
print("\n=== HASIL UJI SENSITIVITAS SCALING (POIN 3) ===")
display(
    scaling_df[
        [
            "Model",
            "Handling",
            "Precision",
            "Recall",
            "F1",
            "FPR",
            "Resample_Time_s",
            "Fit_Time_s",
        ]
    ].round(6)
)

# B. UJI KESTABILAN 5-FOLD DUPLICATE-AWARE (NONE VS CLASS WEIGHT VS RUS PADA XGBOOST & RF)
multifold_results = []
for fold_i, (tr_idx, te_idx) in enumerate(splits):
    print(f"Memproses Fold {fold_i+1}/5 untuk uji kestabilan pembagian data...")
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    for h in ["None", "Class Weight", "RUS"]:
        X_r, y_r, rt = apply_handling(X_tr, y_tr, h, seed=2024)

        # XGBoost
        xgb_m = create_xgb(seed=2024)
        sw = get_balanced_weights(y_r) if h == "Class Weight" else None
        t0 = time.time()
        xgb_m.fit(X_r, y_r, sample_weight=sw)
        ft = time.time() - t0
        res = eval_full(
            xgb_m, y_te=y_te, X_te=X_te, model_name="XGBoost", handling=h,
            split_type=f"Fold_{fold_i}", resample_time=rt, fit_time=ft
        )
        multifold_results.append(res)

mf_df = pd.DataFrame(multifold_results)
mf_df.to_csv("Revisi_5Fold_Stability.csv", index=False)
print("\n=== RATA-RATA ± STD LINTAS 5 FOLDS BEBAS DUPLIKASI (POIN 4) ===")
display(
    mf_df.groupby(["Model", "Handling"])[
        ["Accuracy", "Precision", "Recall", "F1", "FPR", "AP"]
    ]
    .agg(["mean", "std"])
    .round(6)
)

Memproses Uji Scaling | Handling: SMOTE ...
Memproses Uji Scaling | Handling: ADASYN ...
Memproses Uji Scaling | Handling: Tomek Links ...

=== HASIL UJI SENSITIVITAS SCALING (POIN 3) ===


,Model,Handling,Precision,Recall,F1,FPR,Resample_Time_s,Fit_Time_s
0,XGBoost,SMOTE,0.997683,0.999819,0.998750,0.000832,38.815533,33.908036
1,XGBoost,ADASYN,0.993394,0.999864,0.996618,0.002383,210.205593,31.929700
2,XGBoost,Tomek Links,0.997877,0.999397,0.998636,0.000762,678.203108,21.879441


Memproses Fold 1/5 untuk uji kestabilan pembagian data...
Memproses Fold 2/5 untuk uji kestabilan pembagian data...
Memproses Fold 3/5 untuk uji kestabilan pembagian data...
Memproses Fold 4/5 untuk uji kestabilan pembagian data...
Memproses Fold 5/5 untuk uji kestabilan pembagian data...

=== RATA-RATA ± STD LINTAS 5 FOLDS BEBAS DUPLIKASI (POIN 4) ===


Accuracy           Precision              Recall  \
                          mean       std      mean       std      mean   
Model   Handling                                                         
XGBoost Class Weight  0.999098  0.000859  0.998582  0.000415  0.997880   
        None          0.999076  0.000877  0.998650  0.000452  0.997727   
        RUS           0.998837  0.000797  0.998388  0.000473  0.997063   

                                      F1                 FPR            \
                           std      mean       std      mean       std   
Model   Handling                                                         
XGBoost Class Weight  0.003486  0.998228  0.001687  0.000487  0.000161   
        None          0.003532  0.998186  0.001723  0.000464  0.000175   
        RUS           0.003452  0.997722  0.001555  0.000551  0.000173   

                            AP            
                          mean       std  
Model   Handling                          
XGBoost Class Weight  0.999966  0.000023  
        None          0.999966  0.000023  
        RUS           0.999962  0.000021

In [4]:
# ==============================================================================
# EKSPERIMEN REVISI POIN 2: KOMPOSISI JENIS SERANGAN & RECALL PER JENIS SERANGAN
# ==============================================================================
import numpy as np
import pandas as pd

# 1. Ganti nama file di bawah ini sesuai nama 3 file Parquet asli di laptop Anda
files = [
    "Benign-Monday-no-metadata.parquet",  # Sesuaikan nama file Monday Anda
    "DoS-Wednesday-no-metadata.parquet",  # Sesuaikan nama file Wednesday Anda
    "DDoS-Friday-no-metadata.parquet",  # Sesuaikan nama file Friday Anda
]

# Jika Anda sebelumnya memuat dari CSV/Parquet gabungan yang masih ada kolom Label string asli,
# cukup muat file tersebut ke variabel df_raw.
df_list = [pd.read_parquet(f) for f in files]
df_raw = pd.concat(df_list, ignore_index=True)

# Bersihkan spasi pada nama kolom jika ada
df_raw.columns = df_raw.columns.str.strip()

# Lakukan pembersihan missing/infinite values yang sama persis dengan eksperimen awal
df_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
df_raw.dropna(inplace=True)

print("Total baris setelah cleaning:", len(df_raw))

# 2. TABEL KOMPOSISI SETIAP JENIS SERANGAN (UNTUK BAB 2.2 / BAB 3.1)
attack_composition = pd.DataFrame(
    {
        "Count": df_raw["Label"].value_counts(),
        "Percentage (%)": (
            df_raw["Label"].value_counts(normalize=True) * 100
        ).round(4),
    }
)
print("\n=== KOMPOSISI LENGKAP JENIS SERANGAN (POIN 2) ===")
display(attack_composition)

Total baris setelah cleaning: 1265086

=== KOMPOSISI LENGKAP JENIS SERANGAN (POIN 2) ===


,Count,Percentage (%)
Label,,
Benign,943316,74.5654
DoS Hulk,172846,13.6628
DDoS,128014,10.1190
DoS GoldenEye,10286,0.8131
DoS slowloris,5385,0.4257
DoS Slowhttptest,5228,0.4133
Heartbleed,11,0.0009


In [5]:
# ==============================================================================
# MENGHITUNG RECALL PER JENIS SERANGAN PADA DATA UJI (MODEL TETAP BINER)
# ==============================================================================
# Simpan label asli (string) dan label biner (0 = Benign, 1 = Attack)
raw_labels = df_raw["Label"].astype(str).str.strip().copy()
y_binary = np.where(raw_labels.str.upper() == "BENIGN", 0, 1).astype(np.int8)

# Ambil 20 fitur yang digunakan pada penelitian
X_raw20 = df_raw[selected_features].astype(np.float32).copy()

# Gunakan split yang sama dengan eksperimen Anda (misal: Duplicate-Aware Fold 2 atau Stratified Split)
# Di sini kita contohkan pada Stratified Split (random_state=2024) dan Duplicate-Aware Split (Fold 2)
from sklearn.model_selection import train_test_split

X_tr_p2, X_te_p2, y_tr_p2, y_te_p2, raw_tr, raw_te = train_test_split(
    X_raw20, y_binary, raw_labels, test_size=0.20, random_state=2024, stratify=y_binary
)

# Bandingkan Recall per Jenis Serangan untuk: None, Class Weight, SMOTE, ADASYN, RUS, Tomek Links pada XGBoost & RF
per_attack_records = []

for h in ["None", "Class Weight", "SMOTE", "ADASYN", "RUS", "Tomek Links"]:
    X_res, y_res, _ = apply_handling(X_tr_p2, pd.Series(y_tr_p2), h, seed=2024)
    
    # Latih XGBoost
    xgb_p2 = create_xgb(seed=2024)
    sw = get_balanced_weights(pd.Series(y_res)) if h == "Class Weight" else None
    xgb_p2.fit(X_res, y_res, sample_weight=sw)
    preds_xgb = xgb_p2.predict(X_te_p2)
    
    # Hitung Recall untuk setiap sub-kategori serangan di Test Set
    for atk_type in raw_te.unique():
        if atk_type.upper() == "BENIGN":
            continue
        mask = (raw_te == atk_type).values
        total_atk = mask.sum()
        detected_atk = (preds_xgb[mask] == 1).sum()
        recall_atk = detected_atk / total_atk if total_atk > 0 else np.nan
        
        per_attack_records.append({
            "Model": "XGBoost",
            "Handling": h,
            "Attack_Type": atk_type,
            "Test_Support": total_atk,
            "Detected_TP": detected_atk,
            "Recall": round(recall_atk, 6)
        })

per_attack_df = pd.DataFrame(per_attack_records)
pivot_recall = per_attack_df.pivot(index=["Attack_Type", "Test_Support"], columns="Handling", values="Recall")
print("\n=== RECALL PER JENIS SERANGAN (XGBOOST BINER) ===")
display(pivot_recall)


=== RECALL PER JENIS SERANGAN (XGBOOST BINER) ===


,Handling,ADASYN,Class Weight,None,RUS,SMOTE,Tomek Links
Attack_Type,Test_Support,,,,,,
DDoS,25732,0.999845,0.999806,0.999728,0.999845,0.999767,0.999728
DoS GoldenEye,2077,0.999519,0.999037,0.998556,0.999519,0.999519,0.998556
DoS Hulk,34458,0.999942,0.999942,0.999768,0.999913,0.999797,0.999739
DoS Slowhttptest,1040,0.998077,0.997115,0.994231,0.997115,0.998077,0.993269
DoS slowloris,1046,0.996176,0.995220,0.991396,0.996176,0.994264,0.991396
Heartbleed,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
# ==============================================================================
# TAMBAHAN EKSPERIMEN REVISI: RANDOM FOREST UNTUK SEMUA SKENARIO (POIN 1, 3, 4)
# ==============================================================================

# 1. DUPLICATE-AWARE SPLIT UNTUK RANDOM FOREST (SEMUA KONFIGURASI)
rf_rev1_results = []
for h in handlings:
    print(f"Memproses Duplicate-Aware Split (Random Forest) | Handling: {h} ...")
    X_res, y_res, r_time = apply_handling(X_train_da, y_train_da, h, seed=2024)
    
    cw = "balanced" if h == "Class Weight" else None
    rf = create_rf(seed=2024, class_weight=cw)
    t_fit0 = time.time()
    rf.fit(X_res, y_res)
    rf_fit_time = time.time() - t_fit0
    
    rf_rev1_results.append(
        eval_full(
            rf, X_test_da, y_test_da, "Random Forest", h,
            "Duplicate-Aware (Fold 2)", r_time, rf_fit_time
        )
    )

rf_rev1_df = pd.DataFrame(rf_rev1_results)
print("\n=== HASIL DUPLICATE-AWARE SPLIT (RANDOM FOREST SAJA) ===")
display(rf_rev1_df.round(6))


# 2. UJI SENSITIVITAS SCALING UNTUK RANDOM FOREST (SMOTE, ADASYN, TOMEK LINKS)
rf_scaling_results = []
for h in ["SMOTE", "ADASYN", "Tomek Links"]:
    print(f"Memproses Uji Scaling (Random Forest) | Handling: {h} ...")
    X_res_sc, y_res_sc, r_time_sc = apply_handling(X_train_scaled, y_train_da, h, seed=2024)
    
    rf_sc = create_rf(seed=2024)
    t0 = time.time()
    rf_sc.fit(X_res_sc, y_res_sc)
    fit_t = time.time() - t0
    
    rf_scaling_results.append(
        eval_full(
            rf_sc, X_test_scaled, y_test_da, "Random Forest", h,
            "Scaled + Duplicate-Aware", r_time_sc, fit_t
        )
    )

rf_scaling_df = pd.DataFrame(rf_scaling_results)
print("\n=== HASIL UJI SENSITIVITAS SCALING (RANDOM FOREST) ===")
display(rf_scaling_df[["Model", "Handling", "Precision", "Recall", "F1", "FPR", "Resample_Time_s", "Fit_Time_s"]].round(6))


# 3. UJI KESTABILAN 5-FOLD UNTUK RANDOM FOREST (NONE, CLASS WEIGHT, RUS)
rf_multifold_results = []
for fold_i, (tr_idx, te_idx) in enumerate(splits):
    print(f"Memproses Fold {fold_i+1}/5 untuk Random Forest...")
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    for h in ["None", "Class Weight", "RUS"]:
        X_r, y_r, rt = apply_handling(X_tr, y_tr, h, seed=2024)
        
        cw = "balanced" if h == "Class Weight" else None
        rf_m = create_rf(seed=2024, class_weight=cw)
        t0 = time.time()
        rf_m.fit(X_r, y_r)
        ft = time.time() - t0
        
        res = eval_full(
            rf_m, X_te=X_te, y_te=y_te, model_name="Random Forest", handling=h,
            split_type=f"Fold_{fold_i}", resample_time=rt, fit_time=ft
        )
        rf_multifold_results.append(res)

rf_mf_df = pd.DataFrame(rf_multifold_results)
print("\n=== RATA-RATA ± STD 5 FOLDS (RANDOM FOREST) ===")
display(
    rf_mf_df.groupby(["Model", "Handling"])[["Accuracy", "Precision", "Recall", "F1", "FPR", "AP"]]
    .agg(["mean", "std"])
    .round(6)
)

Memproses Duplicate-Aware Split (Random Forest) | Handling: None ...
